In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import time
import sklearn.metrics as mt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import confusion_matrix

# Task 1a)
Define a function *create_dataset()* that 
- takes in input the filename of the full dataset and the names of 2 apps (traffic flows we want to distinguish)
- selects traffic flows of the required apps
- selects the features (statistics of packet size and interarrival time)
- checks if there are flows with infinite or negative values of features
- checks for duplicate flows
- saves flows (features and labels) of the required apps in a .csv file in a subfolder called 'Features' (**the .csv file must be named "apps_flow_dataset_[app1]_[app2].csv" e.g., "apps_flow_dataset_MS_ONE_DRIVE_YOUTUBE.csv")**

Hints: pandas library documentation: https://pandas.pydata.org/pandas-docs/stable/reference/frame.html

Useful functions in Pandas:
- *read_csv* - load csv files content into a dataframe
- *isnull* - check if there are missing value
- *replace* - replace values in a data frame
- *dropna* - check/drop missing values 
- *drop_duplicates* - check/drop duplicates
- *to_csv* - saves a dataframe into a csv file


In [ ]:
def create_dataset(filename, app_names):
# Inputs: - filename: name of the full data file to be read
#         - app_names: list with names of the apps traffic flows which we want to distinguish
# Outputs: - features file (.csv) with the dataset
#           The generated file should be put in a subfolder named "Features"

    # e.g., Min.Packet.Length = min(Fwd.Packet.Length.Min, Bwd.Packet.Length.Min)
    selected_features = [
       'Fwd.Packet.Length.Max', 'Fwd.Packet.Length.Min', 'Fwd.Packet.Length.Mean',
       'Bwd.Packet.Length.Max', 'Bwd.Packet.Length.Min', 'Bwd.Packet.Length.Mean',
       'Flow.Bytes.s',  'Flow.Packets.s',
       'Flow.IAT.Mean', 'Flow.IAT.Max', 'Flow.IAT.Min',
       'Fwd.IAT.Mean',  'Fwd.IAT.Max',  'Fwd.IAT.Min',
       'Bwd.IAT.Mean',  'Bwd.IAT.Max',  'Bwd.IAT.Min',
       'Min.Packet.Length', 'Max.Packet.Length', 'Packet.Length.Mean']

    print('Creating dataset with flows of apps {} and {} from file {}'.format(app_names[0], app_names[1], filename))
    
    feature_folder = 'Features'
    if not os.path.exists(feature_folder):
        os.makedirs(feature_folder)

    
    ############# ADD YOUR CODE BELOW #############
    # Use df as name for the pandas dataframe where to put the dataset
    df = pd.read_csv(filename, delimiter=',')

    # select flows (row-wise) for the given apps ...
    df = df[(df['ProtocolName'] == app_names[0]) | (df['ProtocolName'] == app_names[1])]
    
    # ... store list of labels...
    label_vector = df['ProtocolName'].to_numpy()
    
    # ... and use only the necessary features (column-wise filter)
    df = df[selected_features]
    
    # append column 'Label'
    df['Label'] = label_vector
    
    # example demonstrating that Min.Packet.Length = min(Fwd.Packet.Length.Min, Bwd.Packet.Length.Min)
    #print(df.iloc[0])

    # filter out flows with unreasonable values (doesn't seem to happen)
    for feature in selected_features:
        if df[df[feature] < 0].shape[0] != 0:
            print('Feature {} has negative values in some flows').format(feature)
            df = df[df[feature] >= 0]

    # check if there are NaN, inf (doesn't seem to happen)
    df.replace([np.inf, -np.inf], np.nan, inplace=True) #there is dropna but no dropinf
    if pd.isnull(df).any(axis = None):
        print('There are NaN or inf values')
        df.dropna(inplace=True)

    # check if there are duplicate rows
    if df.duplicated().any(axis = None):
        print('There are duplicated rows')
        df.drop_duplicates(inplace=True)

    ################################################################
    df.to_csv(feature_folder + '/apps_flow_dataset_' + app_names[0] + '_' + app_names[1] + '.csv', index=False)

# Task 1b)
Use function *create_dataset()* with apps ‘MS_ONE_DRIVE’ and ‘YOUTUBE'

(code is already given below)

In [ ]:
apps = ['MS_ONE_DRIVE', 'YOUTUBE']
#apps = ['SKYPE', 'DROPBOX'] #F: alternatively

create_dataset('22_apps_flow_features.csv', apps)


# Task 2a)
Define a function *plot_feature_distribution()* that 
- takes in input names of the 2 apps
- searches for the corresponding flow file in folder "Features" (as created in task 1b)
- plots distribution of the features (one graph per feature, in each graph you should put the distribution of the feature for both apps with two different colors)
- saves the figure(s) in a subfolder called 'Figures'

Hints:
- You can use pandas *df.groupby(label_column_name)[feature_column_name].plot()* function to plot feature "feature_column_name" for all the data points grouped by *label_column_name* 

See documentation at https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.plot.html

In [ ]:
def plot_feature_distribution(app_names):
# Input: - app_names - names of the apps traffic flows of which we want to distinguish
# Output: - plots of the distribution of each feature for 2 apps in the dataset; 
#           figures should be also saved into a subfolder ("Figures")

    print('Plotting feature distribution for apps {} and {}'.format(app_names[0], app_names[1]))
    fig_folder = 'Figures' 
    if not os.path.exists(fig_folder):
        os.makedirs(fig_folder)

    # You should first load data from the file you created at task 1 into a new dataframe 

    ############# ADD YOUR CODE BELOW #############
    
    df = pd.read_csv('Features/apps_flow_dataset_{}_{}.csv'.format(app_names[0], app_names[1]))

    plt.figure(figsize=(15, 60)) #F: comment if you want to have different figures
    for i, column_name in enumerate(df.columns):
        if column_name != 'Label':
            #plt.figure(figsize=(15, 100)) #F: uncomment if you want to have different figures 
            ax = plt.subplot(df.columns.size, 1, i + 1)
            df.groupby('Label')[column_name].plot(kind = 'hist', alpha = 0.5, ax=ax)
            plt.title('Distribution of feature {} for classes {} and {}'.format(column_name, app_names[0], app_names[1]))
            plt.xlabel(column_name)
            plt.legend(loc='best')
            plt.tight_layout(pad=0.4, w_pad=0.5, h_pad=1.0)

            #F: Next 3 lines in case you want to save different figures (you should also uncomment lines above)
            #plt.savefig(fig_folder + '/features_' + app_names[0] + '_' + app_names[1] + '_' + column_name + '.png', bbox_inches='tight')
            #plt.show() 
            #plt.close()
    

    plt.savefig(fig_folder + '/features_' + app_names[0] + '_' + app_names[1] + '.png', bbox_inches='tight')
    #plt.close()

# Task 2b)
Use function *plot_feature_distribution()* to plot distribution of features for  ‘MS_ONE_DRIVE’ and ‘YOUTUBE' traffic categories

(code is already given below)

In [ ]:
plot_feature_distribution(apps)

# Task 3a) 
Define a function *load_dataset()* that 
- takes in input names of the 2 apps
- searches for the corresponding flow file in folder "Features" (as created in task 1b)
- returns numpy arrays (X, y) with features and labels

**N.B. Assign labels so as to generate a binary dataset useful for binary app identification problem (MS_ONE_DRIVE = 0 and YOUTUBE = 1)**

Hint: 
- Use a support dataframe to load data from filename and then select proper columns to create the output numpy arrays
- function *pandas.DataFrame.apply* can be used to convert apps names in 0/1 labels (see https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.apply.html)

In [ ]:
def load_dataset(app_names):
#Inputs: - app_names: names of the apps traffic flows of which we want to distinguish
#Outputs: - X: matrix (numpy.ndarray) of features retrieved from filename
#         - y: column (numpy.ndarray) of labels retrieved from filename

############# ADD YOUR CODE BELOW #############

    feature_df = pd.read_csv('Features/apps_flow_dataset_{}_{}.csv'.format(app_names[0], app_names[1])) #support dataframe

    feature_df['Label'] = feature_df['Label'].apply(lambda x: 0 if x == app_names[0] else 1)

    X = feature_df.drop(columns=['Label']).to_numpy()
    y = feature_df['Label'].to_numpy()

    return X, y

# Task 3b) 
Use function *load_dataset()* to load the dataset created

(code is already given below)


In [ ]:
#apps = [‘MS_ONE_DRIVE’, ‘YOUTUBE'] #F: this is a reminder; app_names has been already defined
X, y = load_dataset(apps)

print(X.shape)
print(y.shape)

# Task 3c)
Do a 80/20 split of the dataset into train/test **with balanced classes** and check consistency between # of positives (i.e., datapoints with y=1) across train/test/full sets.

Hints:
- Use *train_test_split* from *sklearn.model_selection* to split the dataset https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
- Split should be done by maintaining classes proportions between sets

In [ ]:
############# ADD YOUR CODE BELOW #############

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

print(X.shape)
print(y.shape)
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

print('# positives in y = ' + str(np.count_nonzero(y==1)))
print('# positives in y_train = ' + str(np.count_nonzero(y_train==1)))
print('# positives in y_test = ' + str(np.count_nonzero(y_test==1)))

#F: alternatively:
#np.unique(y, return_counts=True)

# Task 4a) 
Define function *train_classifier_logistic()* that performs hyperparameter optimization with 5-fold crossvalidation and training (details below) and returns the trained logistic regression model

Hints:
- Use *KFold* from *sklearn.model_selection* to perform crossvalidation splits (see https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html)
- Use *LogisticRegression* from *sklearn.linear_model* to build/fit a logistic regression classifier (see https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)

In [ ]:
def train_classifier_logistic(X_train, y_train, resfilelogistic): 
#Inputs: - X_train: training set (features)
#        - y_train: training set (ground truth labels)
#        - resfilelogistic: full name (with path) of the results file where to put output of the training/optimization process
#This function should:
#         * Perform logistic regression hyperparameters optimization via 5-fold crossvalidation
#         * Print hyperparameters obtained with crossvalidation in resfilelogistic
#         * Retrain a logistic regression model with best hyperparameters using the entire training set (X_train, y_train)
#         * Print training results (best accuracy and training duration) in resfilelogistic
#         * Return the trained model
#
#Outputs: - updated resfilelogistic with:
#                   1) best hyperparameters obtained with crossvalidation
#                   2) best accuracy obtained during crossvalidation (for the best hyperparameters)
#                   3) duration of training over the entire training set (X_train, y_train)
#         - bestlogisticRegr: model to be returned by the function
#         - best_params: dictionary with best hyperparameters obtained with crossvalidation


    # crossvalidation
    n_split_kfold = 5
    skf = KFold(n_splits=n_split_kfold, shuffle=True, random_state=42)
    
    best_params = {'regularization': 0, 'max_iter': 0, 'score': 0}
    C_reg_range = [1, 10, 100, 1000]
    maxit_range = [1000, 5000, 10000]
    scoreplot = np.zeros([len(C_reg_range),len(maxit_range)])

    t0 = time.time()
    for i, C_reg in enumerate(C_reg_range): #loop over regularization parameter
        
        for j, maxit in enumerate(maxit_range): #loop over maximum iterations
            
            print(f'Testing hyperparameters: regularization: {C_reg}, max_iter: {maxit}')
            
            score = 0
            for train_index, test_index in skf.split(X_train, y_train):
                
                X_train_fold, X_test_fold = X_train[train_index], X_train[test_index]
                y_train_fold, y_test_fold = y_train[train_index], y_train[test_index]
                
                # Instantiate
                logisticRegr = LogisticRegression(C=C_reg, max_iter=maxit, random_state=42)
                
                # Train
                logisticRegr.fit(X_train_fold, y_train_fold)
                
                # Calculate score and cumulate with scores on previous folds
                score += logisticRegr.score(X_test_fold, y_test_fold)
                
            score = score / n_split_kfold
            #print('Score: '+str(score))
            scoreplot[i,j] = score
            
            if score > best_params['score']:
                best_params['score'] = score
                best_params['regularization'] = C_reg
                best_params['max_iter'] = maxit
                #print('New best hyperparameters!!')

    t1 = time.time()
    crossval_time = round(t1 - t0, 3)            
    print('Crossval time [s]: ' + str(crossval_time))
    print('Best hyperparams during crossval: ' + str(best_params))
    
    print(scoreplot)
    
    for p in list(range(scoreplot.shape[1])):
        plt.plot(C_reg_range,scoreplot[:, p], label=f'maxiter={maxit_range[p]}')
        
    plt.xlabel("Regularization parameter 'C'")
    plt.ylabel("Accuracy")
    plt.legend(loc='best')
    plt.show()
    
    training_time = 0 #F: here you will store final training time
    finalscore = 0 #F: here you will store final accuracy
    
    #now retrain with best_params and using the entire training set
    #this part should produce the final model bestlogisticRegr to be returned at the end (together with best_params)
    ############# ADD YOUR CODE BELOW #############
    
    bestlogisticRegr = LogisticRegression(C = best_params['regularization'], max_iter = best_params['max_iter'], random_state = 42)
    t0 = time.time()
    bestlogisticRegr.fit(X_train, y_train) 
    t1 = time.time()
    training_time = round(t1 - t0,3)
    finalscore = bestlogisticRegr.score(X_train, y_train)
    print('Score of the final model: ' + str(finalscore))
    print('Training time [s]: ' + str(training_time))
    
    
    ###############################################
    
    with open(resfilelogistic, 'w') as result_file:
        result_file.write('*** LOGISTIC REGRESSION ***\n')
        result_file.write('*** Crossvalidation results ***\n')
        result_file.write('Best C: {}\n'.format(best_params['regularization']))
        result_file.write('Best number of iterations: {}\n'.format(best_params['max_iter']))
        result_file.write('Best crossvalidation accuracy: {}\n\n'.format(best_params['score']))

        result_file.write('*** Best model results (retrained with entire training set) ***\n')
        result_file.write('Accuracy: {}\n'.format(finalscore))
        result_file.write('Training duration [s]: {}\n'.format(training_time))
    

    return bestlogisticRegr, best_params

# Task 4a')
Test function *train_classifier_logistic()*

(code is already given below)

In [ ]:
res_folder = 'Results'
if not os.path.exists(res_folder):
    os.makedirs(res_folder)
resf = res_folder + '/testlogregr_results.txt'
logreg, best_params_logreg = train_classifier_logistic(X_train, y_train, resf)

# Task 4b)
Consider a new dataset (X_std, y) instead of (X,y) with **standardized** features and repeat train/test split and hyperparameters selection with crossvalidation with the standardized dataset.

Hints:
- Use *StandardScaler()* and *fit_transform()* from *sklearn.preprocessing* to perform feature standardization (mean=0, standard dev = 1)
- Take inspiration from tasks 3c) and  4a') to perform dataset split and training/validation
- To avoid confusion, you may want to name train/test datasets in a different way, e.g., X_train_std, X_test_std, y_train_std, y_test_std

In [ ]:
############# ADD YOUR CODE BELOW #############
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)

print(X_train_std.shape)
print(y_train.shape)
print(X_test_std.shape)
print(y_test.shape)



###############################################
res_folder = 'Results'
if not os.path.exists(res_folder):
    os.makedirs(res_folder)
resf=res_folder + '/testlogregr_results_STD.txt'
logreg_std, best_params_logreg_std = train_classifier_logistic(X_train_std, y_train, resf)

# Task 4c) 
Define function *train_classifier_XGB()* that performs hyperparameter optimization and training for XGB with given hyperparameters space (details below)

Hints: 
- Take inspiration from task 4a)
- Use *XGBClassifier* from *xgboost*

In [ ]:
def train_classifier_XGB(X_train, y_train, resfileXGB): 
#Inputs: - X_train: training set (features)
#        - y_train: training set (ground truth labels)
#        - resfileXGB: full name (with path) of the results file where to put output of the training/optimization process
#This function should:
#         * Perform XGB hyperparameters optimization via 5-fold crossvalidation. 
#         * Use the following XGB hyperparameters with ranges specified below: n_estimators, max_depth, subsample  
#         * Print best hyperparameters obtained with crossvalidation in resfileXGB
#         * Retrain a XGB model with best hyperparameters using the entire training set (X_train, y_train)
#         * Print training results (best accuracy and training duration) in resfileXGB
#         * Return the trained model
#
#Outputs: - updated resfileXGB with:
#                   1) best hyperparameters obtained with crossvalidation
#                   2) best accuracy obtained during crossvalidation (for the best hyperparameters)
#                   3) duration of training over the entire training set (X_train, y_train)
#         - bestXGB: model to be returned by the function
#         - best_params: dictionary with best hyperparameters obtained with crossvalidation
    
    best_params = {'n_estimators': 0, 'max_depth': 0, 'subsample': 0, 'score': 0}
    num_estimators_range = [25, 50, 100]
    max_depth_range = [5, 20, 50]
    subsample_range = [0.7, 0.9, 1]

    ############# ADD YOUR CODE BELOW #############


    # crossvalidation
    n_split_kfold = 5
    skf = KFold(n_splits=n_split_kfold, shuffle=True, random_state=42)
    
    scoreplot = np.zeros([len(num_estimators_range), len(max_depth_range), len(subsample_range)])

    t0 = time.time()
    for i, n_est in enumerate(num_estimators_range): #loop over num_estimators
        for j, max_dep in enumerate(max_depth_range): #loop over max_depth
            for k, subsamp in enumerate(subsample_range): #loop subsample
                
                print(f'Testing hyperparameters: n_estimators: {n_est}, max_depth: {max_dep}, subsample: {subsamp}')
                score = 0
                for train_index, test_index in skf.split(X_train, y_train):
                    X_train_fold, X_test_fold = X_train[train_index], X_train[test_index]
                    y_train_fold, y_test_fold = y_train[train_index], y_train[test_index]
                    
                    xgb = XGBClassifier(verbosity = 0, objective = 'binary:logistic', eval_metric=['error', 'logloss'], 
                                          n_estimators = n_est, max_depth = max_dep, subsample = subsamp)

                    xgb.fit(X_train_fold, y_train_fold)
                    score += xgb.score(X_test_fold, y_test_fold)
                    
                score = score / n_split_kfold
                #print('Score: '+str(score))
                scoreplot[i, j, k] = score

                if score > best_params['score']:
                    best_params['score'] = score
                    best_params['n_estimators'] = n_est
                    best_params['max_depth'] = max_dep
                    best_params['subsample'] = subsamp
                    #print('New best hyperparameters!!')
    t1 = time.time()
    crossval_time = round(t1 - t0,3)            
    print('Crossval time [s]: ' + str(crossval_time))
    print('Best hyperparams: ' + str(best_params))                         
                         
    print(scoreplot)
    #print(scoreplot.shape)

    for p in list(range(scoreplot.shape[0])):
        for q in list(range(scoreplot.shape[1])):
            #print('p,q: ' + str(p)+str(q))
            plt.plot(subsample_range, scoreplot[p,q,:], label=f'n_est={num_estimators_range[p]} - max_dep={max_depth_range[q]} ')
        
    plt.xlabel("Subsample") 
    plt.ylabel("Accuracy")
    plt.legend(loc='best')
    plt.show()

    #now retrain with best_params and using the entire training set
    bestxgb = XGBClassifier(verbosity = 0, objective = 'binary:logistic', eval_metric=['error', 'logloss'], 
                                          n_estimators = best_params['n_estimators'], 
                                          max_depth = best_params['max_depth'], 
                                          subsample = best_params['subsample'])
 
    t0 = time.time()
    bestxgb.fit(X_train, y_train) 
    t1 = time.time()
    training_time = round(t1-t0,3)
    finalscore = bestxgb.score(X_train, y_train)
    print('Score of the final model: ' + str(finalscore))
    print(f'Best n_estimators: {best_params["n_estimators"]}.')
    print(f'Best max_depth: {best_params["max_depth"]}.')
    print(f'Best subsample: {best_params["subsample"]}')
    print('Training time [s]: ' + str(training_time))
    
    
    ###############################################
    
    with open(resfileXGB, 'w') as result_file:
        result_file.write('*** XGB ***\n')
        result_file.write('*** Crossvalidation results ***\n')
        result_file.write('Best n_estimators: {}\n'.format(best_params['n_estimators']))
        result_file.write('Best max_depth: {}\n'.format(best_params['max_depth']))
        result_file.write('Best subsample: {}\n'.format(best_params['subsample']))
        result_file.write('Best crossvalidation accuracy: {}\n\n'.format(best_params['score']))

        result_file.write('*** Best model results (retrained with entire training set) ***\n')
        result_file.write('Accuracy: {}\n'.format(finalscore))
        result_file.write('Training duration [s]: {}\n'.format(training_time))
    
    return bestxgb, best_params

# Task 4d)
Use function *train_classifier_XGB()* to perform hyperparameters selection with crossvalidation for XGB and **considering non-standardized dataset retrieved in task 4b)**

(code is already given below)

In [ ]:
res_folder = 'Results'
if not os.path.exists(res_folder):
    os.makedirs(res_folder)
resf=res_folder + '/testXGB_results.txt'
XGB, best_params_XGB = train_classifier_XGB(X_train, y_train, resf)

# Task 5a)

Perform prediction using the optimized logistic regression **considering standardized dataset and obtained in task 4b)** and XGB **considering non-standardized dataset and obtained in task 4d)**. Then, print the shapes of the predicted vectors.

Hint: 
- Use function *predict()* to use the trained models on the test set

In [ ]:
############# COMPLETE YOUR CODE BELOW #############

y_pred_logreg = logreg_std.predict(X_test_std)
y_pred_XGB = XGB.predict(X_test)

print(y_pred_logreg.shape)
print(y_pred_XGB.shape)

# Task 5b)
Define function *performance_eval()* that takes in input ground truth and predicted labels, prints results in a result file passed in input, and returns global metrics (details below)

Hints:
- Use *confusion_matrix* and other metrics from *sklearn.metrics* https://scikit-learn.org/stable/modules/model_evaluation.html
- To compute global metrics, use *average=‘weighted'* in scikit-learn APIs

In [ ]:
def performance_eval(y_true, y_pred, lab, l_names, resfile):
#Inputs: - y_true: ground-truth labels
#        - y_pred: predicted labels
#        - lab: list of labels (integers) as used during training phase
#        - l_names: list of labels (categories) corresponding to integer labels "lab"; 
#                   can be customized, it's only used for results plotting
#        - resfile: full name (with path) of the results file where to put performance metrics
#This function should:            
#         * Compute Accuracy, Precision (global-weighted and per class), Recall (global-weighted and per class), F1-score (global-weighted and per class)
#         * Write them in result .txt file (resfile passed in input)
#         * Compute confusion matrix (cm) and save it in the same result .txt file (resfile)
#         * Return Accuracy, overall Precision, overall Recall, overall F1-score
#         * Plot of the cm is already given in the code (you need to call the confusion matrix objects as cm and cm_norm)
#Outputs: - updated resfileXGB with:
#                              1) Accuracy, Precision (global-weighted and per class), Recall (global-weighted and per class), F1-score (global-weighted and per class)
#                              2) Confusion matrix (cm) and confusion matrix normalized wrt true labels (cm_norm)
#         - return accuracy, global_precision, global_recall, global_f1score (use these variable names!)
############# ADD YOUR CODE BELOW #############
    
    #Compute metrics and print/write them
    accuracy = mt.accuracy_score(y_true, y_pred)
    precision = mt.precision_score(y_true, y_pred, labels=lab, average=None) #F: average=None gives per-class results
    global_precision = mt.precision_score(y_true, y_pred, labels=lab, average='weighted') 
    recall = mt.recall_score(y_true, y_pred, labels=lab, average=None)
    global_recall = mt.recall_score(y_true, y_pred, labels=lab, average='weighted') 
    f1score = mt.f1_score(y_true, y_pred, labels=lab, average=None)
    global_f1score = mt.f1_score(y_true, y_pred, labels=lab, average='weighted')

    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] #F: normalized wrt true labels (axis=1 sums the elements of one row and sums them; this is done for all the rows independently)

########################################################################
    with open(resfile, 'w') as result_file:
        result_file.write('Results for the TEST SET\n')
        result_file.write('Accuracy: {}\n'.format(accuracy))
        result_file.write('Precision per class: {}\n'.format(precision))
        result_file.write('Global Precision: {}\n'.format(global_precision))
        result_file.write('Recall per class: {}\n'.format(recall))
        result_file.write('Global Recall: {}\n'.format(global_recall))
        result_file.write('F1-score: {}\n'.format(f1score))
        result_file.write('Global F1-score: {}\n'.format(global_f1score))
        
        result_file.write('\nConfusion matrix, without normalization\n')
        np.savetxt(result_file, cm, fmt = '%.2f')

        result_file.write('\nNormalized confusion matrix\n')
        #np.savetxt(result_file, cm_norm, fmt = '%.2f')
        result_file.write(str(cm_norm))

        
        
#F: in the following, we plot the confusion matrices (absolute and normalized one)
# ------------------------------- absolute -------------------------------
        title = 'Confusion matrix'
        fig, ax = plt.subplots()
        im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
        ax.figure.colorbar(im, ax=ax)
        # We want to show all ticks...
        ax.set(xticks=np.arange(cm.shape[1]),
               yticks=np.arange(cm.shape[0]),
               # ... and label them with the respective list entries
               #xticklabels=lab, yticklabels=lab,
               xticklabels=l_names, yticklabels=l_names,
               title=title,
               ylabel='True label',
               xlabel='Predicted label')

        # Rotate the tick labels and set their alignment.
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

        # Loop over data dimensions (#F: i.e., cells in the confusion matrix) and create text annotations. 
        fmt = 'd'
        thresh = cm.max() / 2.
        for w in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, w, format(cm[w, j], fmt),
                        ha="center", va="center",
                        color="white" if cm[w, j] > thresh else "black")
        fig.tight_layout()
        fig.savefig(resfile.replace('.txt','_conf_matrix.png'))
        plt.show()
# ------------------------------- normalized -------------------------------
        title_norm = 'Normalized confusion matrix'
        fig_n, ax_n = plt.subplots()
        im = ax_n.imshow(cm_norm, interpolation='nearest', cmap=plt.cm.Blues)
        ax_n.figure.colorbar(im, ax=ax_n)
        # We want to show all ticks...
        ax_n.set(xticks=np.arange(cm_norm.shape[1]),
               yticks=np.arange(cm_norm.shape[0]),
               # ... and label them with the respective list entries
               #xticklabels=lab, yticklabels=lab,
               xticklabels=l_names, yticklabels=l_names,
               title=title_norm,
               ylabel='True label',
               xlabel='Predicted label')

        # Rotate the tick labels and set their alignment.
        plt.setp(ax_n.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

        # Loop over data dimensions (#F: i.e., cells in the confusion matrix) and create text annotations. 
        fmt = '.2f'
        thresh = cm_norm.max() / 2.
        for w in range(cm_norm.shape[0]):
            for j in range(cm_norm.shape[1]):
                ax_n.text(j, w, format(cm_norm[w, j], fmt),
                        ha="center", va="center",
                        color="white" if cm_norm[w, j] > thresh else "black")
        fig_n.tight_layout()
        fig_n.savefig(resfile.replace('.txt','_conf_matrix_normalized.png'))
        plt.show()

# Now we return the global metrics (Accuracy, Precision, Recall, F1-score) calculated above

    return accuracy, global_precision, global_recall, global_f1score 


# Task 5c)
Call function *performance_eval()* to evaluate the performance of the two models

Hint: 
- Reuse the predicted outputs obtained before (see task 5a)

In [ ]:
#file names to pass as input to performance_eval()
restestfilelogreg = res_folder + '/logreg_test_results.txt'
restestfileXGB = res_folder + '/XGB_test_results.txt'

lbl = [0, 1]
label_names=apps #label names to use as inputs to performance_eval()
############# ADD YOUR CODE BELOW #############


print('************** LOGISTIC REGRESSION **************')
logreg_metrics = performance_eval(y_test, y_pred_logreg, lbl, label_names, restestfilelogreg)
print('Logistic regression metrics (accuracy, global precision, global recall, global f1score): ' +str(logreg_metrics))
print('**************************************************\n\n')

print('************** XGB **************')
XGB_metrics = performance_eval(y_test, y_pred_XGB, lbl, label_names, restestfileXGB)
print('XGB metrics (accuracy, global precision, global recall, global f1score): ' +str(XGB_metrics))

# Appendix

Doing a transformation with PCA and 2 components, we are able to visualize data on a 2D graph 

(code already given below)


In [ ]:
#F: We apply PCA to transform feature space into a 2D features space, so that we can plot on a 2D graph 
from sklearn.decomposition import PCA

X_std = scaler.fit_transform(X)

pca = PCA(n_components=2)
pca.fit(X_std)
X_new = pca.transform(X_std)


print('--------------------')
print(X.shape) #F: size of the original features space
print(X_new.shape) #F: size of the PCA-transformed 2D features space
print(X_new) #F: new features


In [ ]:
#F: We create new arrays, each including data points of one scenario only (either A, B)....
xnewA = [X_new[i,:] for i in range(X_new.shape[0]) if y[i]==0]
xnewB = [X_new[i,:] for i in range(X_new.shape[0]) if y[i]==1]

In [ ]:
#F: ... and transform them in lists, splitting the two components of each data point, one per list (for drawing purposes)
xA=[xnewA[i][0] for i in range(len(xnewA))]
yA=[xnewA[i][1] for i in range(len(xnewA))]
xB=[xnewB[i][0] for i in range(len(xnewB))]
yB=[xnewB[i][1] for i in range(len(xnewB))]

In [ ]:
#F: Finally, we visualize the 2 sets of data points (2 classes) with 2 different colors 
A=plt.scatter(xA,yA,c='red')
B=plt.scatter(xB,yB,c='blue')
plt.legend((A,B),(apps[0],apps[1]), loc='best')

# Task 6) - HOMEWORK (max 1 point)

## Putting things together: multi-class app classification
Repeat previous tasks but considering 3 different apps (Skype, Dropbox, Google):
1. Dataset creation
2. Data visualization
3. Dataset loading
4. Hyperparameters optimization and model training
5. Testing and performance evaluation

